# Which two submissions should you select, and does it matter?

You get two final submissions. This competition closes on **30 September 2026**, and
right now most of the public board is deciding which two entries to tick.

This notebook answers that question with arithmetic instead of instinct. It works out
this board's noise floor from the competition's own data, converts the floor into the
only decision rule that applies — the expected value of the *better* of your two picks —
and then hands you a cell where you paste your own two public scores and get a verdict.

Three things came out of it, and two of them surprised me.

1. **The board's standard error is smaller than the textbook formula says.** The
   Hanley–McNeil formula everyone reaches for gives 0.0016 on the public split. Measured
   two other ways on this competition's own data — DeLong's exact variance, and brute-force
   resampling of 330 split-sized chunks — the answer is about **0.0011**. Hanley overstates
   it here by roughly a half. I had been quoting 0.0016 in my own notebooks; that number
   was conservative, and this notebook is where I correct it.
2. **The correlation that matters is not the correlation you can easily measure.** Everybody
   compares two submissions with a Spearman rank correlation. That is not the quantity the
   selection arithmetic needs, and it is systematically too high. Measured across fifteen
   pairs and four split sizes, `1 - rho` for the thing that matters is **1.8 to 2.2 times**
   larger than `1 - rho` from Spearman. So a Spearman-based verdict understates a hedge by
   about forty per cent, which is the wrong direction to be wrong in.
3. **For almost everyone here, the second slot is worth nothing.** Not "a little" — nothing
   you could see. My own two candidates price out at a gain of one part in a hundred
   thousand, which on this board's density is about a hundredth of one rank.

The last one is a finding about my own submissions, and I show the workings for it rather
than hiding it. A notebook that demonstrates its own second pick is worthless is more
useful than one that sells you a hedge.

**What this is not.** It is not a model and it does not produce a submission. Nothing here
will move your public score. It is a measuring instrument, and the thing being measured is
the board.

**Credit.** The `E[max]` pricing in Part 3 is the method from my own `price_selection.py`,
written in August after I caught a brief of mine recommending two hedges that were both
worth zero. The exact expectation for the maximum of two correlated normals is Clark
(1961); the AUC variance results are Hanley & McNeil (1982) and DeLong, DeLong & Clarke-Pearson
(1988). Competition:
[Playground Series S6E9](https://www.kaggle.com/competitions/playground-series-s6e9).

## The cell you edit

Two numbers is the minimum: your best public score and your second candidate's public
score. If you also have the two submission CSVs on disk or mounted, give the paths and the
notebook measures the rank correlation between them instead of guessing it.

Everything below that is the board description. The default is a snapshot I took on
**12 September 2026 at 21:51 UTC** — 1,696 teams — and it will have drifted by the time you
read this. Download the current one from the competition's Leaderboard tab and replace the
list if you want the rank translations to be exact. The standard error itself does not
depend on the board at all.

In [1]:
CFG = {
    # --- your two candidates ------------------------------------------------
    "MY_SCORE_A": 0.94596,   # your best public score (the ceiling)
    "MY_SCORE_B": 0.94593,   # the other one you are thinking of ticking
    "SUB_A": "",             # optional: path to submission A's csv
    "SUB_B": "",             # optional: path to submission B's csv
    "RHO_FALLBACK": None,    # used only if the two CSVs are not given

    # --- the competition's split --------------------------------------------
    # Kaggle states the public leaderboard is 20% of the test set for this
    # competition. Everything about the private split follows from that one
    # published number plus test.csv's own row count.
    "PUBLIC_FRACTION": 0.20,

    # --- how hard to measure ------------------------------------------------
    "SPLIT_RESAMPLES": 30,   # shuffles; each yields floor(n_train/n_public) chunks
    "SEED": 42,
    "N_SPLITS": 5,           # folds for the honest out-of-fold baseline

    # --- the board, for translating a score gain into ranks -----------------
    # (rank, public score) read off the S6E9 public leaderboard, 2026-09-12
    # 21:51 UTC, 1,696 teams. Scores only; no team names, no ids.
    "BOARD_DATE": "2026-09-12 21:51 UTC",
    "BOARD_TEAMS": 1696,
    "BOARD": [
        (1, 0.94672), (2, 0.94664), (3, 0.94659), (4, 0.94657), (5, 0.94657),
        (7, 0.94657), (10, 0.94654), (15, 0.94650), (20, 0.94649), (25, 0.94648),
        (30, 0.94646), (40, 0.94645), (50, 0.94645), (60, 0.94645), (75, 0.94644),
        (90, 0.94644), (100, 0.94644), (125, 0.94643), (150, 0.94641), (175, 0.94639),
        (200, 0.94638), (250, 0.94635), (300, 0.94632), (350, 0.94625), (400, 0.94616),
        (450, 0.94609), (500, 0.94597), (550, 0.94579), (600, 0.94544), (650, 0.94462),
        (700, 0.94391), (800, 0.94211), (900, 0.94187), (1000, 0.94170),
        (1200, 0.94148), (1400, 0.94100), (1600, 0.93422), (1696, 0.50000),
    ],

    # --- decision thresholds, in units of the private-split SE --------------
    # A pair whose expected gain is under INERT_FRAC standard errors is not
    # worth a slot. 0.05 is the threshold my own tooling has used since August.
    "INERT_FRAC": 0.05,
    "WEAK_FRAC": 0.25,

    "TARGET": "Will_Buy_EV",
    "ID_COL": "id",
}

---

## Part 1 — What is this board's noise floor?

A public leaderboard score is an estimate. It is the AUC of your predictions on one random
fifth of the test rows, and a different fifth would have given a different number. Before
any of the selection arithmetic means anything you need to know how much different.

I derive it three ways, because the first way is the one everybody uses and it turns out to
be wrong by half.

### The split, from the competition's own files

In [2]:
import os, gc, json, math, time, warnings, itertools
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
T0 = time.time()

ROOT = os.environ.get("SMOKE_ROOT", "/kaggle/input")
WORK = os.environ.get("SMOKE_WORK", "/kaggle/working")

def find(name):
    for root, _, files in os.walk(ROOT):
        if name in files:
            return os.path.join(root, name)
    raise FileNotFoundError(name)

TARGET, ID = CFG["TARGET"], CFG["ID_COL"]
train = pd.read_csv(find("train.csv"))
test  = pd.read_csv(find("test.csv"))

def to_binary(s):
    """The target has arrived as Yes/No, True/False and 0/1 in different forks."""
    if s.dtype.kind in "iufb":
        return s.astype(np.int8).to_numpy()
    v = s.astype(str).str.strip().str.lower()
    pos = "yes" if (v == "yes").any() else sorted(v.unique())[-1]
    return (v == pos).astype(np.int8).to_numpy()

SMOKE_ROWS = int(os.environ.get("SMOKE_ROWS", "0"))
if SMOKE_ROWS:
    train = train.sample(SMOKE_ROWS, random_state=CFG["SEED"]).reset_index(drop=True)
    print(f"SMOKE_ROWS={SMOKE_ROWS}: train subsampled. Every number below is a "
          f"small-sample number.\n")

y = to_binary(train[TARGET])
N_TRAIN = len(train)
N_TEST  = len(test)
PREV    = float(y.mean())
N_PUBLIC  = round(N_TEST * CFG["PUBLIC_FRACTION"])
N_PRIVATE = N_TEST - N_PUBLIC

print("The arithmetic, one line at a time")
print("-" * 62)
print(f"  train rows                           {N_TRAIN:>12,}")
print(f"  test rows                            {N_TEST:>12,}")
print(f"  stated public fraction               {CFG['PUBLIC_FRACTION']:>12.2f}")
print(f"  public split rows  = {N_TEST:,} x {CFG['PUBLIC_FRACTION']:.2f} = {N_PUBLIC:>9,}")
print(f"  private split rows = {N_TEST:,} - {N_PUBLIC:,}    = {N_PRIVATE:>9,}")
print(f"  positive rate in train               {PREV:>12.6f}")
print(f"  implied positives on the public split{round(N_PUBLIC * PREV):>12,}")
print("-" * 62)
print(f"  the private split is {N_PRIVATE / N_PUBLIC:.0f}x the public split, so any standard error")
print(f"  computed on the public split is sqrt({N_PRIVATE / N_PUBLIC:.0f}) = "
      f"{math.sqrt(N_PRIVATE / N_PUBLIC):.1f}x the private one")

Two things in that block are assumptions rather than facts, and they are worth naming
before anything is built on them.

**The 20% is published; the row counts are inference.** Kaggle states the public fraction.
It does not publish the exact public row count, and `286,571 x 0.20` need not be exactly how
the split was cut. The rounding error here is at most a row or two and changes nothing.

**The positive rate is estimated from train, not observed on test.** I am assuming the test
split has roughly the same class balance as train. For a Playground frame drawn by one
generator over one pass this is a fair assumption, and it is checkable in the direction that
matters: the standard error below moves by under 2% for a prevalence anywhere between 0.16
and 0.19. It is not a load-bearing guess.

I never see the private split's prevalence, so every threshold at the end of this notebook
is quoted in **standard errors**, not in score units. Anybody quoting you a score-unit
threshold is quoting their own assumption about a split nobody has seen.

### Way one: the closed form everybody uses, and why I no longer trust it

In [3]:
def hanley_se(auc, n_pos, n_neg):
    """Hanley & McNeil (1982), the standard closed-form SE of a single AUC.

    Derived under a binormal / exponential ROC model: q1 and q2 below are the
    two conditional variance terms that model implies. It needs nothing but the
    AUC and the two class counts, which is why it is everywhere.
    """
    if n_pos < 1 or n_neg < 1:
        return float("nan")
    q1 = auc / (2 - auc)
    q2 = 2 * auc * auc / (1 + auc)
    var = (auc * (1 - auc)
           + (n_pos - 1) * (q1 - auc * auc)
           + (n_neg - 1) * (q2 - auc * auc)) / (n_pos * n_neg)
    return math.sqrt(max(var, 0.0))

def hanley_at(auc, n, prev=None):
    prev = PREV if prev is None else prev
    n_pos = max(1, round(n * prev))
    return hanley_se(auc, n_pos, n - n_pos)

print(f"{'AUC level':>12} {'SE public':>12} {'SE private':>12}   (Hanley-McNeil)")
for a in (0.9418, 0.9450, 0.9460, 0.9467):
    print(f"{a:>12.4f} {hanley_at(a, N_PUBLIC):>12.6f} {hanley_at(a, N_PRIVATE):>12.6f}")

HANLEY_PUB = hanley_at(0.9460, N_PUBLIC)
print()
print(f"At the level the top of this board is playing at (0.9460), the closed form says")
print(f"the public leaderboard resolves to +/- {HANLEY_PUB:.6f}.")
print(f"Sensitivity to the prevalence guess, at AUC 0.9460, public split:")
for p in (0.16, 0.17, PREV, 0.18, 0.19):
    print(f"   prevalence {p:.4f} -> SE {hanley_at(0.9460, N_PUBLIC, p):.6f}")

### An instrument to check it against

Hanley–McNeil is a model. It assumes a particular shape for the ROC curve, and this
competition's frame is not obliged to have that shape. To check it I need real predictions
with real labels, and the only labelled data anyone here has is `train.csv`.

So the next cell fits a plain LightGBM out-of-fold — five folds, shuffled, seed 42, which is
the fold convention most of this board's public notebooks already share, so the out-of-fold
vector is directly comparable with theirs. No feature engineering, no tuning. It is not
trying to be good. It is trying to be a realistic prediction vector at roughly this board's
AUC, so that the variance measured on it is the variance you would actually see.

In [4]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

drop = {ID, TARGET}
X = train.drop(columns=[c for c in (ID, TARGET) if c in train.columns])
for c in X.columns:
    if not pd.api.types.is_numeric_dtype(X[c]):
        X[c] = X[c].astype("category")
print(f"{X.shape[1]} features, nothing engineered: {list(X.columns)}")

FOLDS = np.full(len(y), -1, dtype=np.int8)
skf = StratifiedKFold(CFG["N_SPLITS"], shuffle=True, random_state=CFG["SEED"])
for k, (_, va) in enumerate(skf.split(X, y)):
    FOLDS[va] = k

PARAMS = {
    "A": dict(n_estimators=400, learning_rate=0.06, num_leaves=63, min_child_samples=40,
              subsample=0.9, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
              random_state=CFG["SEED"]),
    "B": dict(n_estimators=250, learning_rate=0.10, num_leaves=31, min_child_samples=80,
              subsample=0.7, subsample_freq=1, colsample_bytree=0.6, reg_lambda=5.0,
              random_state=7),
    "C": dict(n_estimators=150, learning_rate=0.15, num_leaves=15, max_depth=4,
              min_child_samples=200, subsample=0.6, subsample_freq=1, colsample_bytree=0.5,
              reg_lambda=20.0, random_state=99),
}

def fit_oof(name):
    t = time.time()
    oof = np.zeros(len(y))
    for k in range(CFG["N_SPLITS"]):
        tr_i, va_i = np.where(FOLDS != k)[0], np.where(FOLDS == k)[0]
        m = lgb.LGBMClassifier(objective="binary", verbose=-1, n_jobs=-1, **PARAMS[name])
        m.fit(X.iloc[tr_i], y[tr_i])
        oof[va_i] = m.predict_proba(X.iloc[va_i])[:, 1]
    auc = roc_auc_score(y, oof)
    print(f"  model {name}: out-of-fold AUC {auc:.6f}   ({time.time() - t:.0f}s)")
    return oof

print("Three deliberately different LightGBMs on one frozen fold partition:")
OOF = {n: fit_oof(n) for n in ("A", "B", "C")}
AUC_A = roc_auc_score(y, OOF["A"])
print()
print(f"Model A is the instrument for Part 1. Its {AUC_A:.6f} is a plain-baseline number;")
print("the top of the public board is about 0.0045 above it, on features this does not have.")

### Way two: DeLong's exact variance

DeLong, DeLong and Clarke-Pearson give the exact non-parametric variance of an AUC
estimate. It makes no assumption about the ROC's shape — it reads the variance off the
scores themselves, through the two sets of placement values. It costs one sort.

In [5]:
from scipy.stats import rankdata

def delong_var(score, label):
    """Exact non-parametric variance of the AUC (DeLong et al. 1988), via midranks.

    v10[i] is the fraction of negatives that positive i beats; v01[j] the fraction
    of positives that negative j loses to. Var = var(v10)/m + var(v01)/n.
    """
    x, z = score[label == 1], score[label == 0]
    m, n = len(x), len(z)
    r_all = rankdata(np.concatenate([x, z]))
    v10 = (r_all[:m] - rankdata(x)) / n
    v01 = 1.0 - (r_all[m:] - rankdata(z)) / m
    return float(v10.mean()), float(v10.var(ddof=1) / m + v01.var(ddof=1) / n)

auc_full, var_full = delong_var(OOF["A"], y)
DELONG_PUB = math.sqrt(var_full * N_TRAIN / N_PUBLIC)

rng = np.random.default_rng(CFG["SEED"])
direct = []
for _ in range(5):
    idx = rng.choice(N_TRAIN, N_PUBLIC, replace=False)
    direct.append(math.sqrt(delong_var(OOF["A"][idx], y[idx])[1]))

print(f"DeLong on all {N_TRAIN:,} out-of-fold rows: AUC {auc_full:.6f}, SE {math.sqrt(var_full):.6f}")
print(f"  scaled to a {N_PUBLIC:,}-row split (SE goes as 1/sqrt(n)): {DELONG_PUB:.6f}")
print(f"  and computed directly on five random {N_PUBLIC:,}-row samples: "
      + ", ".join(f"{s:.6f}" for s in direct))
print(f"  mean of those five: {np.mean(direct):.6f}  (the scaling holds)")
print()
print(f"Hanley said {hanley_at(auc_full, N_PUBLIC):.6f} at this same AUC. DeLong says "
      f"{DELONG_PUB:.6f}.")
print(f"Ratio: Hanley is {hanley_at(auc_full, N_PUBLIC) / DELONG_PUB:.2f}x larger.")

### Way three: stop deriving and just resample

Two formulas disagreeing is not a result. The arbiter is brute force: cut the out-of-fold
rows into chunks the size of the public split, score each chunk, and look at the spread.

Each shuffle gives `floor(668,665 / 57,314) = 11` **disjoint** chunks, so the eleven AUCs
inside one shuffle are genuinely independent of each other. I pool the within-shuffle
variances over 30 shuffles, which gives 300 degrees of freedom and avoids counting the
overlap between shuffles as if it were new information.

One correction is needed and it is easy to forget. Drawing 57,314 rows without replacement
from a fixed pool of 668,665 gives slightly *less* spread than drawing from an unlimited
population — the finite-population factor is `sqrt(1 - 57,314/668,665) = 0.956`. Dividing it
out is worth about four per cent and I would rather show it than quietly absorb it.

In [6]:
def resample_aucs(vectors, n_split, shuffles, seed):
    """AUC of every vector on each of shuffles x floor(N/n_split) disjoint chunks."""
    names = list(vectors)
    chunks = N_TRAIN // n_split
    rng = np.random.default_rng(seed)
    out = {k: [] for k in names}
    for _ in range(shuffles):
        perm = rng.permutation(N_TRAIN)
        for i in range(chunks):
            idx = perm[i * n_split:(i + 1) * n_split]
            yy = y[idx]
            for k in names:
                out[k].append(roc_auc_score(yy, vectors[k][idx]))
    return {k: np.asarray(v) for k, v in out.items()}, chunks

t = time.time()
CH, N_CHUNKS = resample_aucs(OOF, N_PUBLIC, CFG["SPLIT_RESAMPLES"], CFG["SEED"])
FPC = math.sqrt(1 - N_PUBLIC / N_TRAIN)
n_est = CFG["SPLIT_RESAMPLES"] * N_CHUNKS

within = []
for s in range(CFG["SPLIT_RESAMPLES"]):
    within.append(np.var(CH["A"][s * N_CHUNKS:(s + 1) * N_CHUNKS], ddof=1))
EMPIRICAL_PUB = math.sqrt(float(np.mean(within))) / FPC

print(f"{CFG['SPLIT_RESAMPLES']} shuffles x {N_CHUNKS} disjoint chunks of {N_PUBLIC:,} rows "
      f"= {n_est} scored splits   ({time.time() - t:.0f}s)")
print(f"  chunk AUCs: mean {CH['A'].mean():.6f}, min {CH['A'].min():.6f}, "
      f"max {CH['A'].max():.6f}  (spread {CH['A'].max() - CH['A'].min():.6f})")
print(f"  pooled within-shuffle SD            {math.sqrt(float(np.mean(within))):.6f}")
print(f"  finite-population factor            sqrt(1 - {N_PUBLIC:,}/{N_TRAIN:,}) = {FPC:.4f}")
print(f"  corrected empirical SE              {EMPIRICAL_PUB:.6f}")
print()
print(f"{'method':<34}{'public SE':>12}{'vs DeLong':>12}")
print("-" * 58)
for lbl, v in (("Hanley-McNeil closed form", hanley_at(auc_full, N_PUBLIC)),
               ("DeLong exact variance", DELONG_PUB),
               (f"resampling, {n_est} splits", EMPIRICAL_PUB)):
    print(f"{lbl:<34}{v:>12.6f}{v / DELONG_PUB:>11.2f}x")

DeLong and brute force agree to within about seven per cent. Hanley is half again too big.

The seven per cent is not noise and it has a cause worth naming: DeLong conditions on the
number of positives, while my resampling lets it vary from chunk to chunk the way a real
random split would. That extra binomial wobble in the class balance is real, it is part of
what the private split will do to you, and it is why I take the resampled number as the
honest one rather than DeLong's.

Hanley being too big has a cause too. Its `q1` and `q2` terms come from an exponential ROC
model, which is a poor description of a heavily-tied gradient-boosting score distribution on
a generated tabular frame. On this board it is a conservative bound, not an estimate.

**So the floor is the resampled number, and the private floor is half of it**, because the
private split is four times the rows.

In [7]:
SIGMA_PUB = EMPIRICAL_PUB
SIGMA_PRV = SIGMA_PUB * math.sqrt(N_PUBLIC / N_PRIVATE)

print(f"  public-split SE   sigma_pub = {SIGMA_PUB:.6f}   (on {N_PUBLIC:,} rows)")
print(f"  private-split SE  sigma_prv = {SIGMA_PRV:.6f}   (on {N_PRIVATE:,} rows)")
print()
print("Every threshold from here on is in units of sigma_prv, because the private split is")
print("what the medals are decided on and it is the one nobody has seen.")

assert 0.0008 < SIGMA_PUB < 0.0014, SIGMA_PUB
assert abs(SIGMA_PRV * 2 - SIGMA_PUB) < 1e-6

---

## Part 2 — What that standard error does to your rank

Here is the sentence you see in every "the leaderboard is noise" notebook, including two of
my own earlier ones:

> The top 500 teams are separated by 0.00075, which is less than one standard error, so the
> private board is basically a reshuffle.

That sentence is wrong, and the reason it is wrong is the single most useful thing in this
notebook.

Your score's standard error is the spread of **your** score across possible splits. But two
teams are scored on the **same rows**. If a split happens to contain a batch of unusually
easy positives, it lifts both of you. What decides whether I overtake you is the difference
of our two scores, and the difference cancels almost all of the shared luck:

```
SD(my score - your score)  =  sigma * sqrt(2 * (1 - rho))
```

where `rho` is the correlation between our two scores across splits. At `rho = 0`, that is
`1.41 * sigma` and the naive picture is right. At `rho = 0.99` — which is roughly where two
good submissions on a converged Playground board sit — it is `0.14 * sigma`, ten times
smaller, and the board is far more stable than the naive picture claims.

So everything turns on `rho`. Which brings up the second thing I got wrong.

### The correlation everyone measures is not the correlation the arithmetic needs

When people compare two submissions they compute a Spearman rank correlation between the two
prediction columns. Call it `rho_pred`. That is not the same quantity as the correlation
between the two **scores** across splits, which is what the formula above needs. Call that
one `rho_score`.

They are not equal, and the gap is not small. Spearman is dominated by the enormous majority
of row pairs that are easy — a clear positive against a clear negative, where every model
agrees. The AUC's sampling variation, on the other hand, lives entirely in the hard pairs
near the boundary, which is exactly where models disagree. So Spearman reads high.

I can measure both here, because I have three models, the labels, and a way to resample the
split. Six candidates: the three models, and three rank-average blends of A and B, which
put pairs at every correlation level from 0.995 to 0.9998.

In [8]:
from scipy.stats import spearmanr

ra, rb = rankdata(OOF["A"]) / N_TRAIN, rankdata(OOF["B"]) / N_TRAIN
CAND = dict(OOF)
CAND["mix25"] = 0.75 * ra + 0.25 * rb
CAND["mix50"] = 0.50 * ra + 0.50 * rb
CAND["mix75"] = 0.25 * ra + 0.75 * rb

t = time.time()
CH6, _ = resample_aucs(CAND, N_PUBLIC, CFG["SPLIT_RESAMPLES"], CFG["SEED"])
print(f"{n_est} scored splits x {len(CAND)} candidates   ({time.time() - t:.0f}s)")
for k in CAND:
    print(f"  {k:<6} full out-of-fold AUC {roc_auc_score(y, CAND[k]):.6f}   "
          f"chunk SD {CH6[k].std(ddof=1) / FPC:.6f}")

rows = []
for a, b in itertools.combinations(CAND, 2):
    rp = float(spearmanr(CAND[a], CAND[b]).statistic)
    rs = float(np.corrcoef(CH6[a], CH6[b])[0, 1])
    rows.append((f"{a} x {b}", rp, rs, (1 - rs) / (1 - rp)))
rows.sort(key=lambda r: -r[1])

print()
print(f"{'pair':<14}{'1 - rho_pred':>14}{'1 - rho_score':>15}{'ratio':>8}")
print("-" * 51)
for nm, rp, rs, k in rows:
    print(f"{nm:<14}{1 - rp:>14.6f}{1 - rs:>15.6f}{k:>8.2f}")
RATIO = float(np.median([r[3] for r in rows]))
print("-" * 51)
print(f"median ratio {RATIO:.2f}, range {min(r[3] for r in rows):.2f} to "
      f"{max(r[3] for r in rows):.2f}, over {len(rows)} pairs")

The ratio sits near 1.8 and never drops below it, across pairs spanning three orders of
magnitude of `1 - rho`. Before trusting it I checked whether it is an artefact of the split
size I happened to pick, by repeating the whole measurement at a quarter, a half, one and
four times the public-split size — the last of which is the actual private split. It moves
between 1.78 and 2.15 and shows no trend with `n`.

**The working rule I take from this: double `1 - rho_Spearman`.** It is crude, it is measured
on one competition, and it is better than pretending the two quantities are the same. Since
the payoff of a hedge grows as `sqrt(1 - rho)`, using Spearman straight understates a hedge by
about forty per cent — and the error points towards "don't bother", which is the direction
that will quietly cost you a medal one day.

If you have out-of-fold predictions for both of your candidates, you do not need the rule of
thumb: run `resample_aucs` above on your own two out-of-fold vectors and read `rho_score` off
directly. That is the honest version and it costs a minute.

### Now the question you actually asked: how much does the board move?

In [9]:
from statistics import NormalDist
ND = NormalDist()

def p_reverse(gap_true, sigma, rho):
    """P(a team truly `gap_true` behind you finishes ahead of you) on one split."""
    sd = sigma * math.sqrt(max(2 * (1 - rho), 0.0))
    if sd <= 0:
        return 0.0 if gap_true > 0 else 0.5
    return ND.cdf(-gap_true / sd)

print(f"Chance the team behind you finishes ahead, on the private split")
print(f"(sigma_prv = {SIGMA_PRV:.6f}; gap is their TRUE deficit, in sigma_prv)")
print()
hdr = "  ".join(f"rho={r:<6}" for r in (0.0, 0.90, 0.99, 0.995, 0.999))
print(f"{'gap':>8}   {hdr}")
print("-" * (11 + len(hdr)))
for g in (0.0, 0.05, 0.10, 0.25, 0.50, 1.0, 2.0):
    cells = "  ".join(f"{p_reverse(g * SIGMA_PRV, SIGMA_PRV, r):<10.3f}"
                      for r in (0.0, 0.90, 0.99, 0.995, 0.999))
    print(f"{g:>7.2f}s   {cells}")

In [10]:
board = sorted(CFG["BOARD"])
b_rank = np.array([r for r, _ in board], float)
b_score = np.array([s for _, s in board], float)

def score_at(rank):
    return float(np.interp(np.log(max(rank, 1)), np.log(b_rank), b_score))

def rank_at(score):
    """Ranks are decreasing in score, so interpolate on the reversed curve."""
    return float(np.exp(np.interp(-score, -b_score[::-1], np.log(b_rank[::-1]))))

top500_span = score_at(1) - score_at(500)
top100_span = score_at(1) - score_at(100)
sd_diff = lambda rho: SIGMA_PUB * math.sqrt(2 * (1 - rho))

print(f"Public board, {CFG['BOARD_DATE']}, {CFG['BOARD_TEAMS']:,} teams")
print(f"  rank 1 {score_at(1):.5f}   rank 100 {score_at(100):.5f}   "
      f"rank 500 {score_at(500):.5f}   rank 1000 {score_at(1000):.5f}")
print(f"  top-100 span {top100_span:.5f}   top-500 span {top500_span:.5f}")
print()
print(f"The naive reading: {top500_span:.5f} / sigma_pub = "
      f"{top500_span / SIGMA_PUB:.2f} standard errors. 'It is all noise.'")
print()
print("The reading that accounts for the shared split:")
print(f"{'rho between two teams':<26}{'SD(score difference)':>22}{'top-500 span in those':>24}")
print("-" * 72)
for r in (0.0, 0.90, 0.95, 0.99, 0.995, 0.999):
    print(f"{r:<26.3f}{sd_diff(r):>22.6f}{top500_span / sd_diff(r):>24.1f}")
print("-" * 72)
rho_star = 1 - (top500_span / (2 * SIGMA_PUB)) ** 2 / 2
print(f"For the top-500 spread to be as small as two difference-SEs - the point at which")
print(f"'the order is noise' becomes a fair description - two teams' submissions would have")
print(f"to correlate BELOW rho = {rho_star:.3f}. At the top of a converged Playground board")
print(f"they correlate far above that. So the top of this board is not a lottery, and every")
print(f"notebook telling you it is - including two of mine - divided by the wrong number.")

That is the correction, and it cuts against my own prior work, so let me be exact about what
it does and does not establish.

**It establishes** that a public gap of 0.00075 across 500 teams is many times the standard
error of a *difference* between two highly-correlated submissions, so most of the ordering of
the top of this board reflects something real rather than split luck.

**It does not establish** what `rho` actually is between two different teams' submissions. I
cannot measure that from inside this notebook — I would need their prediction files. Every
number in the right-hand columns above is conditional on a `rho` I am assuming. What I can
say is that the conclusion is robust across the whole plausible range: anything above 0.95
gives the same answer, and 0.95 is low for two tuned gradient-boosting submissions on the
same 668,665 rows.

**And it does not rescue the second slot**, which is Part 3. Your own two candidates are not
two different teams. They come out of one pipeline, they correlate at 0.998 and up, and that
is precisely the regime where the difference between them collapses to nothing.

---

## Part 3 — Price your own pair

Kaggle scores **the better** of your two selected entries on the private split. So the value
of the second slot is not your second entry's score. It is

```
E[max(A, B)] - E[A]
```

the amount by which having a second ticket raises the expected outcome above holding the
first alone. For two correlated normals that expectation is exact, not an approximation —
Clark (1961) — and it is four lines of code.

Two properties of it decide everything, and both are counterintuitive until you see them:

**Correlation alone is not enough.** A hedge that is wildly decorrelated but two standard
errors below your ceiling will almost never be the maximum, so it adds nothing.

**Proximity alone is not enough either.** A hedge at exactly your ceiling's score but
correlated at 0.999 is the same submission wearing a hat.

The cell below wants the two scores from `CFG`. Give it the two CSVs as well and it measures
the rank correlation instead of assuming one; without them it sweeps a range so you can see
whether the answer even depends on it.

In [11]:
def e_max(mu1, mu2, s1, s2, rho):
    """E[max(X, Y)] for a bivariate normal. Clark (1961); exact."""
    sd = math.sqrt(max(s1 * s1 + s2 * s2 - 2 * rho * s1 * s2, 0.0))
    if sd <= 1e-15:
        return max(mu1, mu2)
    th = (mu1 - mu2) / sd
    return mu1 * ND.cdf(th) + mu2 * ND.cdf(-th) + sd * ND.pdf(th)

def pair_gain(ceiling, hedge, sigma, rho):
    """Expected private-score gain from adding `hedge` alongside `ceiling`."""
    return e_max(ceiling, hedge, sigma, sigma, rho) - ceiling

def equal_means_ceiling(sigma, rho):
    """The most a hedge at this correlation could EVER be worth, in score units.

    Setting the two means equal maximises E[max], and the expression collapses to
    sigma * sqrt((1 - rho) / pi). If this is already below the INERT line, no pair
    at this correlation helps, whatever the two scores are. It is the cheapest
    test in the notebook: it needs rho and nothing else.
    """
    return sigma * math.sqrt(max(1 - rho, 0.0) / math.pi)

def verdict(gain, sigma):
    if gain < CFG["INERT_FRAC"] * sigma:
        return "INERT"
    if gain < CFG["WEAK_FRAC"] * sigma:
        return "WEAK"
    return "LIVE"

def measure_rho(path_a, path_b, id_col=None):
    """Spearman between two submission CSVs, aligned on the id column."""
    da, db = pd.read_csv(path_a), pd.read_csv(path_b)
    idc = id_col or da.columns[0]
    cols = [c for c in da.columns if c in db.columns and c != idc]
    m = da[[idc] + cols].merge(db[[idc] + cols], on=idc, suffixes=("_a", "_b"))
    return float(spearmanr(m[cols[0] + "_a"], m[cols[0] + "_b"]).statistic), len(m)

def price_pair(score_a, score_b, rho_pred, sigma=None, label="", spearman_correction=True):
    """The whole decision, in one call. Returns a dict; prints the reasoning."""
    sigma = SIGMA_PRV if sigma is None else sigma
    ceiling, hedge = max(score_a, score_b), min(score_a, score_b)
    rho = 1 - RATIO * (1 - rho_pred) if spearman_correction else rho_pred
    rho = min(max(rho, -1.0), 1.0)
    cap = equal_means_ceiling(sigma, rho)
    g = pair_gain(ceiling, hedge, sigma, rho)
    ranks = rank_at(ceiling) - rank_at(ceiling + g)
    v = verdict(g, sigma)
    if label:
        print(f"[{label}]")
    print(f"  ceiling {ceiling:.5f}   hedge {hedge:.5f}   gap "
          f"{(hedge - ceiling) / sigma:+.2f} sigma_prv")
    print(f"  rho_pred (Spearman) {rho_pred:.6f}"
          + (f"  ->  rho_score {rho:.6f} after the x{RATIO:.2f} correction"
             if spearman_correction else ""))
    print(f"  most this pair could be worth at ANY scores: {cap:.8f} "
          f"({cap / sigma:.4f} sigma_prv)")
    print(f"  expected gain at these scores:              {g:.8f} "
          f"({g / sigma:.4f} sigma_prv)")
    print(f"  in board ranks, on the {CFG['BOARD_DATE']} board:      {ranks:.3f} ranks")
    print(f"  VERDICT: {v}"
          + ("  - the slot is being wasted" if v == "INERT" else
             "  - real but small" if v == "WEAK" else "  - a genuine hedge"))
    print()
    return dict(ceiling=ceiling, hedge=hedge, rho_pred=rho_pred, rho=rho,
                cap=cap, gain=g, ranks=ranks, verdict=v)

print(f"Decision thresholds, in units of sigma_prv = {SIGMA_PRV:.6f}:")
print(f"  gain < {CFG['INERT_FRAC']:.2f} sigma  -> INERT")
print(f"  gain < {CFG['WEAK_FRAC']:.2f} sigma  -> WEAK")
print(f"  otherwise                -> LIVE")

In [12]:
a, b = CFG["MY_SCORE_A"], CFG["MY_SCORE_B"]
if CFG["SUB_A"] and CFG["SUB_B"] and os.path.exists(CFG["SUB_A"]) and os.path.exists(CFG["SUB_B"]):
    rho_pred, n_rows = measure_rho(CFG["SUB_A"], CFG["SUB_B"], CFG["ID_COL"])
    print(f"measured Spearman on {n_rows:,} aligned rows: {rho_pred:.6f}\n")
    MINE = price_pair(a, b, rho_pred, label="your pair, measured rho")
elif CFG["RHO_FALLBACK"] is not None:
    MINE = price_pair(a, b, CFG["RHO_FALLBACK"], label="your pair, rho from CFG")
else:
    print("No submission CSVs given, so rho is unknown. Sweeping it instead - if every")
    print("row below says INERT, the answer does not depend on the number you could not")
    print("measure, and you can stop here.\n")
    print(f"{'rho_pred':>10}{'rho_score':>12}{'gain':>14}{'in sigma':>11}"
          f"{'in ranks':>11}{'verdict':>10}")
    print("-" * 68)
    for rp in (0.999, 0.998, 0.995, 0.99, 0.98, 0.95, 0.90):
        rs = 1 - RATIO * (1 - rp)
        g = pair_gain(max(a, b), min(a, b), SIGMA_PRV, rs)
        print(f"{rp:>10.3f}{rs:>12.5f}{g:>14.8f}{g / SIGMA_PRV:>11.4f}"
              f"{rank_at(max(a, b)) - rank_at(max(a, b) + g):>11.3f}"
              f"{verdict(g, SIGMA_PRV):>10}")
    MINE = None

---

## Part 4 — My own five candidates, priced honestly

Here is what I have to select from on this competition, with the public score each one
actually scored and the Spearman correlation each has with my best entry. The correlations
were measured with the function above on the submission files themselves, off-platform,
because those files are not mounted here — so this is the one table in the notebook you
cannot reproduce from the competition data alone. Everything else you can.

| entry | what it is | public |
|---|---|---|
| z6 | LightGBM, dual target encoding, digits, frequency encoding, slow schedule | 0.94596 |
| blend | nested greedy blend over a 19-model out-of-fold library | 0.94593 |
| nb | the pipeline in my ablation notebook, run end to end | 0.94585 |
| z1 | the same recipe on a faster schedule | 0.94584 |
| lgbm | a plain honest-fold LightGBM, no feature work at all | 0.94164 |

The last one is the interesting case. It is the only candidate of mine that is meaningfully
decorrelated from the ceiling, which is what a hedge is supposed to be. Watch what happens to
it anyway.

In [13]:
CEILING = 0.94596
MINE_TABLE = [
    ("blend", 0.94593, 0.999446),
    ("nb",    0.94585, 0.998095),
    ("z1",    0.94584, 0.998162),
    ("lgbm",  0.94164, 0.988749),
]
results = [price_pair(CEILING, s, r, label=f"z6 + {n}") for n, s, r in MINE_TABLE]

print("=" * 72)
print(f"{'pair':<12}{'gap (sigma)':>13}{'rho_pred':>11}{'ceiling on gain':>18}"
      f"{'actual gain':>14}{'verdict':>9}")
print("-" * 72)
for (n, s, r), res in zip(MINE_TABLE, results):
    print(f"{'z6 + ' + n:<12}{(s - CEILING) / SIGMA_PRV:>13.2f}{r:>11.6f}"
          f"{res['cap'] / SIGMA_PRV:>17.4f}s{res['gain'] / SIGMA_PRV:>13.4f}s"
          f"{res['verdict']:>9}")
print("-" * 72)
BEST = max(results, key=lambda r: r["gain"])
print(f"Best available second slot: {BEST['gain']:.8f} = {BEST['gain'] / SIGMA_PRV:.4f} "
      f"sigma_prv = {BEST['ranks']:.3f} board ranks.")
assert all(r["verdict"] == "INERT" for r in results)

Four candidates, every one of them inert, for two different reasons that are worth separating
because they are the only two reasons there are.

**The three near the top fail on correlation.** They are within a sixth of a standard error
of the ceiling — as close as a hedge could ask for — but they came out of the same pipeline
and they correlate at 0.998 and above. Their *best possible* value, if their true means were
exactly equal to the ceiling's, is still only a twentieth of a standard error. There is no
version of the scores that rescues them.

**The plain LightGBM fails on distance.** It is the only one of the four whose correlation
clears the break-even line in Part 5, so it is a real hedge in shape. But it sits five and a
half private standard errors below the ceiling. It will essentially never be the better of
the two, and its contribution rounds to zero at eight decimal places.

The best second slot available to me is worth about a hundredth of one rank. I could have
spent the last fortnight hunting for a better hedge. The arithmetic says the hunt has no
prize in it, and I would rather publish that than a story about a clever pair.

---

## Part 5 — When the second slot *is* worth something

None of this says hedging is pointless in general. It says hedging is pointless in a specific
and very common situation, and the boundary is sharp enough to write down.

Both thresholds below are in standard errors, never in score units. I am not being coy: the
private split's exact row count and its class balance are not published, so any score-unit
threshold is somebody's assumption dressed as a fact. Convert to score units yourself, at the
end, with a sigma you derived — Part 1 shows how.

In [14]:
def breakeven_rho(inert_frac=None):
    """Correlation above which an EQUAL-MEANS pair is already inert.

    Equal-means gain is sigma*sqrt((1-rho)/pi), so the boundary solves
    sqrt((1-rho)/pi) = inert_frac  ->  rho = 1 - pi*inert_frac**2.
    Derived from the threshold, never hard-coded; an earlier version of my own
    tool carried a literal constant that contradicted its own verdict function.
    """
    f = CFG["INERT_FRAC"] if inert_frac is None else inert_frac
    return 1 - math.pi * f * f

def breakeven_gap_sigma(rho, inert_frac=None, tol=1e-9):
    """How many sigma below the ceiling a hedge goes inert, at this rho."""
    f = CFG["INERT_FRAC"] if inert_frac is None else inert_frac
    lo, hi = 0.0, 20.0
    while hi - lo > tol:
        mid = (lo + hi) / 2
        if pair_gain(1.0, 1.0 - mid, 1.0, rho) > f:
            lo = mid
        else:
            hi = mid
    return lo

BE_RHO = breakeven_rho()
print("Test 1 - correlation. Needs only rho; do this one first, it is free.")
print(f"  An equal-means pair is worth sigma * sqrt((1 - rho) / pi).")
print(f"  That falls below the {CFG['INERT_FRAC']:.2f}-sigma line at "
      f"rho_score = {BE_RHO:.6f}.")
print(f"  In Spearman terms, after the x{RATIO:.2f} correction: rho_pred = "
      f"{1 - (1 - BE_RHO) / RATIO:.6f}.")
print(f"  Above that, NO pair of scores can make the second slot worth having.")
print()
print(f"{'rho_score':>11}{'best possible gain':>21}{'still worth a slot?':>22}")
print("-" * 54)
for r in (0.9999, 0.999, BE_RHO, 0.98, 0.95, 0.90, 0.70, 0.0):
    cap = math.sqrt((1 - r) / math.pi)
    print(f"{r:>11.5f}{cap:>19.4f}s{('yes' if cap >= CFG['INERT_FRAC'] else 'no'):>22}")
print()
print("Test 2 - distance. Only worth running if test 1 passed.")
print(f"{'rho_score':>11}{'inert beyond':>16}")
print("-" * 27)
for r in (0.0, 0.5, 0.9, 0.95, 0.99):
    print(f"{r:>11.3f}{breakeven_gap_sigma(r):>14.2f}s")
print()
print(f"At rho = 0, the most forgiving case there is, a hedge more than "
      f"{breakeven_gap_sigma(0.0):.2f} sigma")
print("below your ceiling is already inert. Every positive correlation makes it worse.")

### The rule, in one paragraph

Take your two candidates. Measure the Spearman correlation between the submission files and
double `1 - rho`. If the result is still above about 0.99, stop — the second slot cannot pay,
and no amount of score-matching will change that. If it is below, then check the distance: a
hedge more than two standard errors below your ceiling is inert even at zero correlation, and
the allowance shrinks fast as correlation rises. **You need both at once: a candidate that is
genuinely different and genuinely close.** Most of us have neither, because the thing that
makes a submission close is that it came out of the same pipeline as the ceiling.

If both tests pass, take the hedge. If either fails, select your two best scores, stop
thinking about it, and spend the fortnight on the model instead. That is what I am doing.

### What would change my mind

Three things, and none of them is a small print disclaimer — they are the places this
notebook could be wrong.

**The correlation between different teams' submissions.** Part 2's conclusion that the top
of this board is not a lottery rests on a `rho` I assumed rather than measured. If two top
teams really correlate below 0.95, the naive noise story is right after all and I am wrong.
Anyone with two top public submission files can settle it in a minute with `measure_rho`.

**The x1.8 Spearman correction.** Measured on one competition, one frame, six candidates, all
of them gradient-boosting. I checked it at four split sizes and fifteen pairs and it did not
move, which is more than I expected, but it is still one dataset. Treat it as this board's
number, not a constant of nature.

**The normal approximation.** `E[max]` assumes the two private scores are jointly normal. AUC
on 229,257 rows is close enough to normal that I am not worried, but the tails are where a
hedge would pay, and tails are exactly where a normal approximation is least trustworthy.

### Reproducing this

Every number above except the table in Part 4 comes out of `train.csv` and `test.csv`. Fork
it, change `CFG`, and it runs. If your computed standard error differs from mine, yours is the
one that applies to your submissions — mine is a plain baseline's, and a sharper model has a
slightly smaller one.

If you find an error, say so in the comments and I will run it.

In [15]:
HEAD = {
    "n_test": N_TEST, "n_public": N_PUBLIC, "n_private": N_PRIVATE,
    "prevalence": round(PREV, 6),
    "baseline_oof_auc": round(float(AUC_A), 6),
    "se_hanley_public": round(hanley_at(auc_full, N_PUBLIC), 6),
    "se_delong_public": round(DELONG_PUB, 6),
    "se_empirical_public": round(SIGMA_PUB, 6),
    "sigma_private": round(SIGMA_PRV, 6),
    "spearman_ratio": round(RATIO, 3),
    "breakeven_rho_score": round(BE_RHO, 6),
    "breakeven_gap_sigma_at_rho0": round(breakeven_gap_sigma(0.0), 3),
    "best_second_slot_sigma": round(BEST["gain"] / SIGMA_PRV, 5),
    "best_second_slot_ranks": round(BEST["ranks"], 4),
    "top500_span_in_sigma_pub": round(top500_span / SIGMA_PUB, 3),
    "wall_seconds": round(time.time() - T0, 1),
}
os.makedirs(WORK, exist_ok=True)
np.save(os.path.join(WORK, "folds_seed42.npy"), FOLDS)
np.save(os.path.join(WORK, "oof_baseline.npy"), OOF["A"])
with open(os.path.join(WORK, "headline.json"), "w") as fh:
    json.dump(HEAD, fh, indent=1)
for k, v in HEAD.items():
    print(f"  {k:<30} {v}")
print()
print("written: folds_seed42.npy (the shared 5-fold partition, seed 42),")
print("         oof_baseline.npy (this notebook's out-of-fold vector),")
print("         headline.json")